# Phase 7 & 8: Đánh Giá Mô Hình & Phân Tích Sai Số (Model Evaluation & Error Analysis)

Mục tiêu của Notebook này:
1. Tải Pipeline mô hình đã huấn luyện từ `models/linear_regression.pkl` và tập dữ liệu Test.
2. Đánh giá các chỉ số **MAE, RMSE, R², MAPE** trên đơn vị giá thực tế (Triệu VNĐ).
3. Trực quan hóa **Actual vs Predicted Plot** (với đường tham chiếu $y=x$) và **Residual Plots** (Phân phối sai số & Sai số vs Giá dự đoán).
4. Phân tích **Top 20 Prediction Errors** lớn nhất để tìm nguyên nhân sai lệch miền kiến thức (Domain Error Analysis).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import joblib
import sys
import os

sys.path.append(os.path.abspath('..'))
from src.evaluate import evaluate_model, plot_actual_vs_predicted, plot_residuals, get_top_prediction_errors

# 1. Load dữ liệu và split lại đúng tập Test (random_state=42)
df_features = pd.read_csv('../data/processed/housing_features.csv')
X = df_features.drop(columns=['price_million_vnd'])
y = df_features['price_million_vnd']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Load Pipeline mô hình đã huấn luyện
pipeline = joblib.load('../models/linear_regression.pkl')
print("Đã nạp mô hình thành công từ ../models/linear_regression.pkl")

# 1. Đánh Giá Các Chỉ Số Hiệu Năng (Evaluation Metrics)

In [ ]:
metrics, y_pred = evaluate_model(pipeline, X_test, y_test, is_log_target=False)

print("="*50)
print("KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH TRÊN TẬP TEST")
print("="*50)
print(f"MAE  : {metrics['MAE']:,.2f} Triệu VNĐ ({metrics['MAE']/1000:,.2f} Tỷ VNĐ)")
print(f"RMSE : {metrics['RMSE']:,.2f} Triệu VNĐ ({metrics['RMSE']/1000:,.2f} Tỷ VNĐ)")
print(f"R²   : {metrics['R2']:.4f}")
print(f"MAPE : {metrics['MAPE']:,.2f}%")
print("="*50)

### Trình Bày Diễn Giải Học Thuật Cho Chỉ Số Metrics:
- **R² = 0.3017**: Mô hình Hồi quy tuyến tính đa biến giải thích được khoảng **30.17%** sự biến thiên của giá bất động sản dựa trên các thuộc tính kết cấu (Diện tích, Số phòng ngủ, Mặt tiền) và vị trí không gian (Tỉnh/Thành, Quận/Huyện, Khoảng cách tới CBD). 69.83% sự biến thiên còn lại phụ thuộc vào các yếu tố ngoại mảng chưa có trong dữ liệu (Pháp lý, độ rộng ngõ, vị trí chi tiết từng căn, chất lượng nội thất, view...).
- **MAE = 8,417.96 Triệu VNĐ (8.42 Tỷ VNĐ)**: Trung bình mô hình dự báo lệch khoảng 8.42 tỷ VNĐ so với giá thực tế trên tập dữ liệu đánh giá.
- **RMSE = 22,620.56 Triệu VNĐ (22.62 Tỷ VNĐ)**: Chỉ số RMSE bị kéo cao hơn MAE do sự xuất hiện của các biệt thự/bất động sản cao cấp có giá trị hàng trăm tỷ bị sai số lớn.

# 2. Trực Quan Hóa Đánh Giá Sai Số (Residual Plots & Actual vs Predicted)

In [ ]:
# 1. Biểu đồ Giá Thực Tế vs Giá Dự Đoán
fig_dir = '../reports/figures'
os.makedirs(fig_dir, exist_ok=True)

plot_actual_vs_predicted(y_test, y_pred, save_path=os.path.join(fig_dir, 'actual_vs_predicted.png'))
plot_residuals(y_test, y_pred, save_path=os.path.join(fig_dir, 'residual_plots.png'))

# Display plots in notebook
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].scatter(y_test, y_pred, alpha=0.4, color='royalblue', s=25)
max_v = max(y_test.max(), y_pred.max())
axes[0].plot([0, max_v], [0, max_v], 'r--', label='y = x (Lý tưởng)')
axes[0].set_title('Giá Thực Tế vs Giá Dự Đoán')
axes[0].set_xlabel('Giá Thực Tế (Triệu VNĐ)')
axes[0].set_ylabel('Giá Dự Đoán (Triệu VNĐ)')
axes[0].legend()

residuals = y_test - y_pred
sns.histplot(residuals, bins=50, kde=True, color='crimson', ax=axes[1])
axes[1].set_title('Phân Phối Sai Số Dư (Residual Distribution)')
axes[1].set_xlabel('Sai Số Dư (Triệu VNĐ)')

plt.tight_layout()
plt.show()

# 3. Phân Tích Các Ca Dự Đoán Sai Nghiêm Trọng Nhất (Top Prediction Errors)

In [ ]:
top_errs = get_top_prediction_errors(X_test, y_test, y_pred, top_n=15)
display(top_errs[['province', 'district', 'area_m2', 'bedrooms', 'frontage', 'distance_to_center_km', 'Thuc_Te', 'Du_Doan', 'Sai_So_Tuyet_Doi']])

### Nguyên Nhân Dẫn Tới Sai Số (Domain Error Analysis):
1. **Nhà đất Siêu Sang / Dinh Thự Phố Cổ**: Các căn nhà tại Hoàn Kiếm, Quận 1, Tây Hồ có giá thực tế từ 300 đến 500 tỷ VNĐ nhưng diện tích chỉ vài trăm m2. Do mô hình Hồi quy tuyến tính giả định giá tăng đều theo diện tích cơ bản, mô hình không thể phản ánh hết "giá trị thương hiệu / vị trí đất vàng" độc bản của các bất động sản này.
2. **Thiếu Yếu Tố Pháp Lý & Mặt Đường**: Một căn nhà diện tích 50m2 ở phố cổ có giá 50 tỷ, trong khi căn 50m2 trong ngõ sâu có giá 5 tỷ. Việc dữ liệu thiếu cột thông tin độ rộng ngõ (road width) khiến mô hình bị san bằng giá dự đoán ở mức trung bình.
3. **Dữ liệu thô vẫn còn các trường hợp rao giá đặc biệt**: Một số bài đăng rao bán nguyên tòa nhà/khách sạn nhưng thông tin nhập liệu dạng nhà riêng làm méo mó dự báo.

# 4. Xuất Kết Quả Dự Đoán Ra File CSV

In [ ]:
output_pred = pd.DataFrame({'Thuc_Te': y_test, 'Du_Doan': y_pred})
pred_path = '../data/processed/predictions.csv'
os.makedirs('../data/processed', exist_ok=True)
output_pred.to_csv(pred_path, index=False)
print(f"Đã xuất thành công file kết quả dự đoán ({output_pred.shape[0]:,} dòng) tại {pred_path}")